# Qualitative Ranking Comparison

Compare the ranked neighbors produced by the CBIR, CF, MM, and joint checkpoints for the same held-out catalogue. The first code cell contains the project/data paths, catalogue family, optional catalogue IDs, and checkpoint locations; update these for your machine and selected experiment before running the notebook. Figures are written under `figures/qualitative_rankings/<catalogue-family>/`.

In [ ]:

import os
import sys
import random
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from torchvision import transforms

# -----------------------------------------------------------------------------
# Paths and selection
# -----------------------------------------------------------------------------
PROJECT_ROOT = "/nas-ctm01/homes/dsviana/retrieval"
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

DATASET_PATH = "/nas-ctm01/datasets/private/CINDERELLA/moe_organized_retrieval_CINDERELLA/multimodal/pre_post"
CACHE_DIR = os.path.join(PROJECT_ROOT, "cache", "multimodal", "pre_post")

CATALOGUE_TYPE = "Fair"  # "Excl" or "Fair"
SPLIT_RATIO = 0.8
SEED = 10

# Put explicit catalogue IDs here. Example: CATALOGUE_IDS = [12, 47, 103]
# Leave empty to select one random catalogue from the evaluation split.
CATALOGUE_IDS: List[int] = []

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "figures", "qualitative_rankings", CATALOGUE_TYPE)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Separate checkpoint for each family, matching the current training scripts.
CBIR_MODEL_PATH = os.path.join(PROJECT_ROOT, "checkpoints", "cbir", CATALOGUE_TYPE, "best_model.pt")
MM_MODEL_PATH = os.path.join(PROJECT_ROOT, "checkpoints", "mm", CATALOGUE_TYPE, "best_model.pt")
CF_MODEL_PATH = os.path.join(PROJECT_ROOT, "checkpoints", "cf", CATALOGUE_TYPE, "NCF", "best_model.pt")
JOINT_MODEL_PATH = os.path.join(PROJECT_ROOT, "checkpoints", "joint", CATALOGUE_TYPE, "best_model.pt")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("Output directory:", OUTPUT_DIR)


In [ ]:
from utils.cbir.cbir_evaluation import comp_pairwise_distance as comp_pairwise_distance_cbir
from utils.catalogue_utils import (
    Catalogue,
    build_catalogues,
    split_catalogues_canonical,
    parse_filename,
)
from utils.cache_preprocessing import get_or_create_cached_image
from utils.image_preprocessing import transform_fn, pad_img
from utils.mm.mm_evaluation import comp_pairwise_distance as comp_pairwise_distance_mm
from utils.mm.mm_data import (
    normalize_tabular_if_needed,
    prepare_multimodal_datasets_from_catalogues,
    tabular_vec_from_img_path,
)
from utils.cf.cf_data import prepare_cf_datasets
from utils.cf.cf_visualization import predict_catalogue_ranking_cf
from models.joint import JointCFMMModel
from utils.joint.joint_data import prepare_joint_cf_mm_datasets
from models.image_encoders import GC_ViT_224
from models.multimodal import MultimodalEncoder
from models.collaborative import FM_model, NN_model, NCF_model


def load_model_state(model: torch.nn.Module, path: str) -> torch.nn.Module:
    """Load either a plain state_dict or a checkpoint containing model_state_dict."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    state = torch.load(path, map_location=device)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    print(f"Loaded: {path}")
    return model


def load_image_tensor(path: str) -> torch.Tensor:
    """Use the same cached preprocessing as training when available."""
    tensor = get_or_create_cached_image(path, CACHE_DIR)
    if tensor is not None:
        return tensor
    return transform_fn(path)


def catalogue_key(catalogue: Catalogue) -> str:
    return f"{catalogue.catalogue_type}_{catalogue.catalogue_id}"


def plot_ranked_catalogue(
    catalogue: Catalogue,
    ranked_paths: List[str],
    save_path: str,
    show: bool = False,
    include_metadata: bool = True,
) -> None:
    """
    Save one model-specific ranking figure.

    The figure intentionally does NOT contain the model name: the dissertation
    caption can identify the model, as requested. The filename remains model-specific.
    """
    if not ranked_paths:
        raise ValueError("The predicted ranking is empty.")

    gt_rank_by_path = {path: idx + 1 for idx, path in enumerate(catalogue.neighbors)}
    num_neighbors = len(ranked_paths)
    neighbor_rows = 2
    neighbor_cols = max(1, (num_neighbors + 1) // neighbor_rows)

    fig = plt.figure(figsize=(2.5 + 2 * neighbor_cols, 4.5))
    gs_main = GridSpec(1, 2, figure=fig, width_ratios=[1.5, neighbor_cols], wspace=0.02)
    gs_left = GridSpecFromSubplotSpec(
        3, 1,
        subplot_spec=gs_main[0, 0],
        height_ratios=[0.1, 1, 0.15],
        hspace=0.02,
    )

    title_ax = fig.add_subplot(gs_left[0, 0])
    title_ax.text(
        0.5, 0.5,
        f"Catalogue\n{catalogue.catalogue_type}_{catalogue.catalogue_id}",
        ha="center", va="center", fontsize=10, fontweight="bold",
    )
    title_ax.axis("off")

    query_ax = fig.add_subplot(gs_left[1, 0])
    query_img = transforms.Resize((224, 224))(pad_img(catalogue.query))
    query_ax.imshow(query_img)
    query_ax.set_title("Query", fontsize=10, pad=3)
    query_ax.axis("off")

    if include_metadata:
        q_meta = parse_filename(os.path.basename(catalogue.query))
        if q_meta:
            q_text = "\n".join([
                str(q_meta["age"]),
                str(q_meta["height"]),
                str(q_meta["weight"]),
                str(q_meta["bra_cup"]),
                str(q_meta["bra_size"]),
            ])
            query_ax.text(
                0.05, 0.95, q_text,
                transform=query_ax.transAxes,
                fontsize=7,
                verticalalignment="top",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.7),
            )

    subtitle_ax = fig.add_subplot(gs_left[2, 0])
    if include_metadata:
        subtitle_ax.text(
            0.5, 0.5,
            "Legend:\nage (years)\nheight (cm)\nweight (kg)\nbra cup\nbra size",
            ha="center", va="center", fontsize=8, style="italic",
        )
    subtitle_ax.axis("off")

    gs_right = GridSpecFromSubplotSpec(
        neighbor_rows,
        neighbor_cols,
        subplot_spec=gs_main[0, 1],
        wspace=0.0,
        hspace=0.0,
    )

    for idx, path in enumerate(ranked_paths):
        row = idx // neighbor_cols
        col = idx % neighbor_cols
        if row >= neighbor_rows:
            break

        pred_rank = idx + 1
        gt_rank = gt_rank_by_path[path]
        title_color = "green" if pred_rank == gt_rank else "red"

        ax = fig.add_subplot(gs_right[row, col])
        ax.imshow(transforms.Resize((224, 224))(pad_img(path)))
        ax.set_title(f"Pred {pred_rank} | GT {gt_rank}", fontsize=8, pad=3, color=title_color)
        ax.axis("off")

        if include_metadata:
            meta = parse_filename(os.path.basename(path))
            if meta:
                text = "\n".join([
                    str(meta["age"]),
                    str(meta["height"]),
                    str(meta["weight"]),
                    str(meta["bra_cup"]),
                    str(meta["bra_size"]),
                ])
                ax.text(
                    0.05, 0.95, text,
                    transform=ax.transAxes,
                    fontsize=6,
                    verticalalignment="top",
                    bbox=dict(boxstyle="round", facecolor="white", alpha=0.7),
                )

    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)
    print("Saved:", save_path)


def predict_image_only(model: torch.nn.Module, catalogue: Catalogue) -> List[str]:
    model.eval()
    with torch.no_grad():
        q = load_image_tensor(catalogue.query).unsqueeze(0).to(device)
        n = torch.stack([load_image_tensor(p) for p in catalogue.neighbors]).to(device)

        q_emb = F.normalize(model(q), p=2, dim=1)
        n_emb = F.normalize(model(n), p=2, dim=1)
        distances = comp_pairwise_distance_cbir(
            q_emb.expand_as(n_emb), n_emb, distance_metric="euclidean"
        ).detach().cpu().numpy()

    order = np.argsort(distances)
    return [catalogue.neighbors[i] for i in order]


def predict_multimodal(
    model: MultimodalEncoder,
    catalogue: Catalogue,
    tabular_stats: Dict[str, np.ndarray],
) -> List[str]:
    q_tab = tabular_vec_from_img_path(catalogue.query)
    if q_tab is None:
        raise ValueError(f"Could not parse metadata for {catalogue.query}")
    q_tab = normalize_tabular_if_needed(q_tab, tabular_stats)

    n_tabs = []
    for path in catalogue.neighbors:
        tab = tabular_vec_from_img_path(path)
        if tab is None:
            raise ValueError(f"Could not parse metadata for {path}")
        n_tabs.append(normalize_tabular_if_needed(tab, tabular_stats))

    model.eval()
    with torch.no_grad():
        q_img = load_image_tensor(catalogue.query).unsqueeze(0).to(device)
        n_img = torch.stack([load_image_tensor(p) for p in catalogue.neighbors]).to(device)
        q_tab_t = torch.tensor(q_tab, dtype=torch.float32, device=device).unsqueeze(0)
        n_tab_t = torch.tensor(np.stack(n_tabs), dtype=torch.float32, device=device)

        q_emb = model(q_img, q_tab_t)
        n_emb = model(n_img, n_tab_t)
        distances = comp_pairwise_distance_mm(
            q_emb.expand_as(n_emb), n_emb, distance_metric="cosine"
        ).detach().cpu().numpy()

    order = np.argsort(distances)
    return [catalogue.neighbors[i] for i in order]


def predict_joint_cf(
    model: JointCFMMModel,
    catalogue: Catalogue,
    tabular_stats: Dict[str, np.ndarray],
) -> List[str]:
    """Joint ranking = CF relevance scores, sorted from highest to lowest."""
    q_tab = tabular_vec_from_img_path(catalogue.query)
    if q_tab is None:
        raise ValueError(f"Could not parse metadata for {catalogue.query}")
    q_tab = normalize_tabular_if_needed(q_tab, tabular_stats)

    n_tabs = []
    for path in catalogue.neighbors:
        tab = tabular_vec_from_img_path(path)
        if tab is None:
            raise ValueError(f"Could not parse metadata for {path}")
        n_tabs.append(normalize_tabular_if_needed(tab, tabular_stats))

    model.eval()
    with torch.no_grad():
        n = len(catalogue.neighbors)
        q_img = load_image_tensor(catalogue.query)
        q_img_batch = torch.stack([q_img] * n).to(device)
        item_img_batch = torch.stack([load_image_tensor(p) for p in catalogue.neighbors]).to(device)
        q_tab_batch = torch.tensor(
            np.repeat(q_tab[None, :], n, axis=0),
            dtype=torch.float32,
            device=device,
        )
        item_tab_batch = torch.tensor(
            np.stack(n_tabs),
            dtype=torch.float32,
            device=device,
        )
        scores = model((q_img_batch, item_img_batch, q_tab_batch, item_tab_batch))
        scores = scores.detach().cpu().numpy().reshape(-1)

    order = np.argsort(-scores)
    return [catalogue.neighbors[i] for i in order]


In [ ]:
# -----------------------------------------------------------------------------
# Select catalogue(s) ONLY from the canonical evaluation split
# -----------------------------------------------------------------------------
all_catalogues = build_catalogues(DATASET_PATH)
train_catalogues, test_catalogues = split_catalogues_canonical(
    all_catalogues,
    split_ratio=SPLIT_RATIO,
    seed=SEED,
    catalogue_type=CATALOGUE_TYPE,
)

if CATALOGUE_IDS:
    by_id = {c.catalogue_id: c for c in test_catalogues}
    missing = [cid for cid in CATALOGUE_IDS if cid not in by_id]
    if missing:
        raise ValueError(
            f"These catalogue IDs are not in the {CATALOGUE_TYPE} evaluation split: {missing}"
        )
    selected_catalogues = [by_id[cid] for cid in CATALOGUE_IDS]
else:
    rng = random.SystemRandom()
    even_catalogues = [
        catalogue for catalogue in test_catalogues
        if len(catalogue.neighbors) >= 6
        and len(catalogue.neighbors) % 2 == 0
    ]
    if not even_catalogues:
        raise ValueError(
            "No evaluation catalogues have at least 8 candidates and an even candidate count."
        )
    selected_catalogues = [rng.choice(even_catalogues)]

print(f"Evaluation catalogues available: {len(test_catalogues)}")
print("Selected:", [catalogue_key(c) for c in selected_catalogues])

# Store only ranking paths, not models, so GPU memory can be released after each model.
rankings: Dict[str, Dict[str, List[str]]] = {
    catalogue_key(c): {} for c in selected_catalogues
}


In [ ]:

# -----------------------------------------------------------------------------
# 1) Image-only GC ViT - Euclidean ranking
# -----------------------------------------------------------------------------
cbir_model = load_model_state(GC_ViT_224(), CBIR_MODEL_PATH)

for catalogue in selected_catalogues:
    rankings[catalogue_key(catalogue)]["image_only"] = predict_image_only(cbir_model, catalogue)

del cbir_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:

# -----------------------------------------------------------------------------
# 2) Standalone multimodal model - cosine ranking
#    Configuration mirrors the current main_mm.py reference run.
# -----------------------------------------------------------------------------
_, _, mm_tabular_stats = prepare_multimodal_datasets_from_catalogues(
    train_catalogues,
    test_catalogues,
    normalize_tabular=True,
    image_transform=transform_fn,
    cache_dir=CACHE_DIR,
    use_cache=True,
    include_catalogue_type_feature=False,
)

mm_model = MultimodalEncoder(
    image_encoder=GC_ViT_224(),
    tabular_in_features=5,
    tabular_embedding_dim=64,
    fusion_embedding_dim=64,
    tabular_nhead=1,
    tabular_num_layers=1,
    tabular_encoder_type="mlp",
    fusion_nhead=1,
    fusion_num_layers=1,
    fusion_mode="cross_attention",
)
mm_model = load_model_state(mm_model, MM_MODEL_PATH)

for catalogue in selected_catalogues:
    rankings[catalogue_key(catalogue)]["multimodal"] = predict_multimodal(
        mm_model, catalogue, mm_tabular_stats
    )

del mm_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:

# -----------------------------------------------------------------------------
# 3) Standalone NCF collaborative filtering WITH metadata
#    The image-only CF ablation is intentionally not visualized.
# -----------------------------------------------------------------------------
cf_image_encoder = GC_ViT_224()
cf_bundle = prepare_cf_datasets(
    data_dir=DATASET_PATH,
    image_encoder=cf_image_encoder,
    split_ratio=SPLIT_RATIO,
    seed=SEED,
    catalogue_type=CATALOGUE_TYPE,
    device=device,
    image_transform=transform_fn,
    cache_dir=CACHE_DIR,
    use_cache=True,
    normalize_tabular=True,
    model_variant="NCF",
    image_embedding_mode="precomputed",
)

# The backbone is only needed to reconstruct the precomputed embedding map.
del cf_image_encoder
if torch.cuda.is_available():
    torch.cuda.empty_cache()

cf_config = {
    "device": device,
    "dropout": 0.1,
    "image_embedding_dim": cf_bundle.image_embedding_dim,
    "projection_dim": 64,
    "k": 8,
    "lay_1": 256,
    "lay_2": 128,
    "lay_3": 64,
    "lay_4": 32,
    "tabular_encoder_dim": 64,
    "tabular_nhead": 1,
    "tabular_num_layers": 1,
    "tabular_encoder_type": "mlp",
    "ncf_fusion_dim": 64,
    "ncf_fusion_heads": 2,
    "ncf_fusion_layers": 2,
    "ncf_dropout": 0.1,
}
cf_fm = FM_model(cf_config)
cf_nn = NN_model(cf_config)
cf_model = NCF_model(
    cf_config,
    nn_model=cf_nn,
    fm_model=cf_fm,
    fusion_mode="cross_attention",
)
cf_model = load_model_state(cf_model, CF_MODEL_PATH)

for catalogue in selected_catalogues:
    ranked = predict_catalogue_ranking_cf(
        model=cf_model,
        catalogue=catalogue,
        metadata=cf_bundle.metadata,
        device=device,
        tabular_stats=cf_bundle.tabular_stats,
        model_variant="NCF",
    )
    rankings[catalogue_key(catalogue)]["cf"] = [path for path, _score in ranked]

del cf_model, cf_fm, cf_nn
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# -----------------------------------------------------------------------------
# 4) Joint MM-CF model
#    IMPORTANT: visualise the joint CF relevance ranking, not the auxiliary MM ranking.
# -----------------------------------------------------------------------------
joint_image_encoder = GC_ViT_224()
joint_bundle = prepare_joint_cf_mm_datasets(
    data_dir=DATASET_PATH,
    image_encoder=joint_image_encoder,
    split_ratio=SPLIT_RATIO,
    seed=SEED,
    catalogue_type=CATALOGUE_TYPE,
    device=device,
    image_transform=transform_fn,
    cache_dir=CACHE_DIR,
    use_cache=True,
    normalize_tabular=True,
    image_embedding_mode="encoded_at_runtime",
    tabular_dim=5,
    tabular_embedding_dim=64,
    fusion_embedding_dim=64,
    tabular_nhead=1,
    tabular_num_layers=1,
    fusion_nhead=1,
    fusion_num_layers=1,
    fusion_mode="cross_attention",
)

joint_model = JointCFMMModel(
    image_embedding_dim=joint_bundle.image_embedding_dim,
    tabular_dim=5,
    tabular_embedding_dim=64,
    fusion_embedding_dim=64,
    tabular_nhead=1,
    tabular_num_layers=1,
    tabular_encoder_type="transformer",
    fusion_nhead=1,
    fusion_num_layers=1,
    hidden_dim=128,
    dropout=0.1,
    image_encoder=joint_image_encoder,
    device=device,
    cf_model_variant="NCF",
    cf_use_fused_input=True,
    cf_projection_dim=64,
    cf_fm_k=8,
    cf_lay_1=256,
    cf_lay_2=128,
    cf_lay_3=64,
    cf_lay_4=32,
    cf_ncf_fusion_heads=2,
    cf_ncf_fusion_layers=2,
    cf_ncf_fusion_dim=64,
    cf_ncf_dropout=0.1,
    fusion_mode="cross_attention",
    freeze_image=False,
    freeze_tabular=False,
)
joint_model = load_model_state(joint_model, JOINT_MODEL_PATH)

for catalogue in selected_catalogues:
    rankings[catalogue_key(catalogue)]["joint"] = predict_joint_cf(
        joint_model,
        catalogue,
        joint_bundle.tabular_stats,
    )

del joint_model, joint_image_encoder
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# -----------------------------------------------------------------------------
# Export FOUR SEPARATE image files per catalogue
# -----------------------------------------------------------------------------
model_file_names = {
    "image_only": "image_only",
    "multimodal": "multimodal",
    "cf": "cf_ncf",
    "joint": "joint",
}

for catalogue in selected_catalogues:
    key = catalogue_key(catalogue)
    for model_key, file_stub in model_file_names.items():
        ranked_paths = rankings[key][model_key]

        output_path = os.path.join(
            OUTPUT_DIR,
            f"{key}_{file_stub}.png",
        )
        plot_ranked_catalogue(
            catalogue,
            ranked_paths,
            save_path=output_path,
            show=False,
            include_metadata=True,
        )
